# 🩺 Nidaan — TrOCR Fine-Tuning (Colab T4)

### 🚀 Clean & Verified Pipeline
- Uses fixed dependencies compatible with Colab's Python environment.
- Cleanly separates model architectural config and generation settings.
- Uses lazy PyTorch Dataset loading to stay under **1.5 GB RAM**.

### 📋 Instructions:
1. `Runtime` → `Change runtime type` → **T4 GPU** → Save.
2. Upload `RxHandBD.zip` using the Files panel (left sidebar).
3. Click `Runtime` → `Run all`.

In [ ]:
# ── CELL 1: Install Dependencies ────────────────────────────────────────
!pip install -q evaluate jiwer albumentations sentencepiece tiktoken

import torch
assert torch.cuda.is_available(), '❌ GPU not found! Go to Runtime → Change runtime type → T4 GPU'
print(f'✅ GPU: {torch.cuda.get_device_name(0)}')
print(f'✅ CUDA Version: {torch.version.cuda}')
import transformers
print(f'✅ Transformers: {transformers.__version__}')

In [ ]:
# ── CELL 2: Unzip RxHandBD Dataset ──────────────────────────────────────
import os, zipfile

ZIP_PATH = '/content/RxHandBD.zip'
EXTRACT_DIR = '/content/RxHandBD'

assert os.path.exists(ZIP_PATH), (
    '❌ RxHandBD.zip not found!\n'
    'Upload it via the Files panel on the left.'
)

if not os.path.exists(EXTRACT_DIR):
    print('📦 Extracting RxHandBD.zip...')
    with zipfile.ZipFile(ZIP_PATH, 'r') as z:
        z.extractall('/content/')
    print('✅ Extraction finished.')
else:
    print('✅ RxHandBD directory already exists.')

train_dir = os.path.join(EXTRACT_DIR, 'Train_Set')
test_dir  = os.path.join(EXTRACT_DIR, 'Test_Set')
train_csv = os.path.join(EXTRACT_DIR, 'Train_Label.csv')
test_csv  = os.path.join(EXTRACT_DIR, 'Test_Labels.csv')

print(f'✅ Train images: {len(os.listdir(train_dir))}')
print(f'✅ Test images : {len(os.listdir(test_dir))}')

In [ ]:
# ── CELL 3: Load Data & Initialize Model Cleanly ────────────────────────
import pandas as pd
import numpy as np
from PIL import Image
import albumentations as A
from torch.utils.data import Dataset
from transformers import TrOCRProcessor, VisionEncoderDecoderModel, GenerationConfig

train_df = pd.read_csv(train_csv)
test_df  = pd.read_csv(test_csv)

train_df['path'] = train_df['Images'].apply(lambda f: os.path.join(train_dir, str(f)))
test_df['path']  = test_df['Images'].apply(lambda f: os.path.join(test_dir,  str(f)))

# Keep only rows where images exist on disk
train_df = train_df[train_df['path'].apply(os.path.exists)].dropna().reset_index(drop=True)
test_df  = test_df[test_df['path'].apply(os.path.exists)].dropna().reset_index(drop=True)

print(f'✅ Filtered samples: {len(train_df)} train | {len(test_df)} test')

# Data augmentation pipeline
aug_pipeline = A.Compose([
    A.Rotate(limit=6, p=0.4),
    A.RandomBrightnessContrast(brightness_limit=0.12, contrast_limit=0.12, p=0.4),
    A.GaussNoise(var_limit=(5, 20), p=0.3),
])

# Load TrOCR-small model and processor
MODEL_ID = 'microsoft/trocr-small-handwritten'
print(f'🔄 Loading {MODEL_ID}...')
processor = TrOCRProcessor.from_pretrained(MODEL_ID)
model = VisionEncoderDecoderModel.from_pretrained(MODEL_ID)

# 1. Cleanly purge any legacy generation kwargs that trigger HuggingFace ValueError
if hasattr(model.config, '_get_generation_parameters'):
    for k in list(model.config._get_generation_parameters().keys()):
        setattr(model.config, k, None)

# 2. Explicitly assign token IDs on model.config for Seq2SeqTrainer loss calculation
pad_id   = processor.tokenizer.pad_token_id
start_id = processor.tokenizer.cls_token_id
eos_id   = processor.tokenizer.sep_token_id

model.config.pad_token_id           = pad_id
model.config.decoder_start_token_id = start_id
model.config.eos_token_id           = eos_id
model.config.decoder.pad_token_id   = pad_id
model.config.vocab_size             = model.config.decoder.vocab_size

# 3. Configure dedicated GenerationConfig for evaluation
model.generation_config = GenerationConfig(
    decoder_start_token_id = start_id,
    pad_token_id           = pad_id,
    eos_token_id           = eos_id,
    max_length             = 32,
    early_stopping         = True,
    no_repeat_ngram_size   = 3,
    length_penalty         = 2.0,
    num_beams              = 4,
)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = model.to(device)
print(f'✅ Model initialized on {device.upper()}')

In [ ]:
# ── CELL 4: Streaming PyTorch Dataset (Zero RAM Overflow) ────────────────
class RxHandDataset(Dataset):
    def __init__(self, df, processor, max_target_length=32, is_train=True):
        self.df = df
        self.processor = processor
        self.max_target_length = max_target_length
        self.is_train = is_train

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image_path = row['path']
        text = str(row['Text']).strip()

        # Load and resize dynamically on demand
        image = Image.open(image_path).convert('RGB')
        image = image.resize((384, 384), Image.BILINEAR)

        if self.is_train:
            arr = np.array(image)
            arr = aug_pipeline(image=arr)['image']
            image = Image.fromarray(arr)

        pixel_values = self.processor(image, return_tensors='pt').pixel_values.squeeze(0)

        # Tokenize target text
        labels = self.processor.tokenizer(
            text,
            padding='max_length',
            max_length=self.max_target_length,
            truncation=True,
            return_tensors='pt'
        ).input_ids.squeeze(0)

        # Replace pad tokens with -100 so CrossEntropy loss ignores padding
        labels[labels == self.processor.tokenizer.pad_token_id] = -100

        return {'pixel_values': pixel_values, 'labels': labels}

train_dataset = RxHandDataset(train_df, processor, is_train=True)
eval_dataset  = RxHandDataset(test_df,  processor, is_train=False)
print(f'✅ Streaming datasets ready: {len(train_dataset)} train, {len(eval_dataset)} eval.')

In [ ]:
# ── CELL 5: Training with Seq2SeqTrainer ────────────────────────────────
import evaluate as ev
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments, default_data_collator
import gc

gc.collect()
torch.cuda.empty_cache()

# Final verification of model.config tokens
model.config.decoder_start_token_id = processor.tokenizer.cls_token_id
model.config.pad_token_id           = processor.tokenizer.pad_token_id
model.config.eos_token_id           = processor.tokenizer.sep_token_id
model.config.decoder.pad_token_id   = processor.tokenizer.pad_token_id

cer_metric = ev.load('cer')

def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids

    pred_str = processor.batch_decode(pred_ids, skip_special_tokens=True)
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id
    label_str = processor.batch_decode(label_ids, skip_special_tokens=True)

    cer = cer_metric.compute(predictions=pred_str, references=label_str)
    return {'cer': round(cer, 4)}

training_args = Seq2SeqTrainingArguments(
    output_dir                  = './nidaan_trocr_ckpts',
    num_train_epochs            = 8,
    per_device_train_batch_size = 8,
    per_device_eval_batch_size  = 8,
    gradient_accumulation_steps = 2,       # Effective batch size = 16
    fp16                        = True,    # 16-bit mixed precision for GPU
    learning_rate               = 5e-5,
    warmup_steps                = 150,
    weight_decay                = 0.01,
    eval_strategy               = 'epoch',
    save_strategy               = 'epoch',
    save_total_limit            = 1,       # Keep only best checkpoint to save disk
    load_best_model_at_end      = True,
    metric_for_best_model       = 'cer',
    greater_is_better           = False,
    predict_with_generate       = True,
    logging_steps               = 50,
    dataloader_num_workers      = 2,
    report_to                   = 'none',
)

trainer = Seq2SeqTrainer(
    model           = model,
    args            = training_args,
    train_dataset   = train_dataset,
    eval_dataset    = eval_dataset,
    data_collator   = default_data_collator,
    compute_metrics = compute_metrics,
)

print('⚡ Starting Training on T4 GPU...')
print('⏳ Estimated time: ~45 to 60 minutes.')
trainer.train()

# Save trained model & processor
output_dir = './nidaan-trocr-final'
trainer.save_model(output_dir)
processor.save_pretrained(output_dir)
print(f'\n🎉 Training complete! Model saved to {output_dir}')

In [ ]:
# ── CELL 6: Sample Predictions on Unseen Test Prescriptions ─────────────
print('🧪 Testing 10 random samples from test set:')
model.eval()

sample_indices = np.random.choice(len(test_df), 10, replace=False)
for i, idx in enumerate(sample_indices):
    item = eval_dataset[idx]
    pixel_values = item['pixel_values'].unsqueeze(0).to(device)

    with torch.no_grad():
        generated_ids = model.generate(pixel_values, max_new_tokens=32)

    pred_text = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
    true_text = test_df.iloc[idx]['Text']
    is_match  = '✅' if pred_text.strip().lower() == str(true_text).strip().lower() else '🔍'

    print(f'Sample {i+1}: True: "{true_text}" | Pred: "{pred_text}" {is_match}')

In [ ]:
# ── CELL 7: Package & Download Trained Model ────────────────────────────
import os, shutil
from google.colab import files

model_dir = './nidaan-trocr-final'
zip_name = 'nidaan-trocr-final'

print('📦 Packaging trained model files into zip...')
shutil.make_archive(zip_name, 'zip', '.', model_dir)

zip_file = f'{zip_name}.zip'
zip_size_mb = os.path.getsize(zip_file) / (1024 * 1024)
print(f'✅ Successfully created {zip_file} ({zip_size_mb:.1f} MB)')

print('⬇️ Downloading model to your laptop...')
files.download(zip_file)